# Modelo estrella: `fact_results`

Este notebook arma un **esquema en estrella** a partir de la tabla `results` de Ergast (`f1db`).

`results` es el grano natural de un resultado de un piloto en una carrera: una fila = un piloto + una carrera.

## Dimensiones (solo las necesarias)

`results` tiene cuatro claves foráneas. Esas son las únicas dimensiones:

| Dimensión | Origen | Por qué existe |
|---|---|---|
| `dim_drivers` | `drivers` | `results.driverId` |
| `dim_constructors` | `constructors` | `results.constructorId` |
| `dim_status` | `status` | `results.statusId` |
| `dim_races` | `races` + `circuits` + `seasons` | `results.raceId` |

`dim_races` **desnormaliza** circuito y temporada (mismo criterio que `productlines` dentro de `dim_products` en el ETL de Glue). Así el modelo sigue siendo estrella y no copo de nieve.

No se crean:

- `dim_circuits` / `dim_seasons` como tablas aparte (entrarían por `races`, no por `results`)
- hechos de `qualifying`, `lapTimes`, `pitStops`, `driverStandings`, `constructorStandings`, `constructorResults` (otro grano, otros hechos)

## Hecho

`fact_results` conserva las **claves** hacia las dimensiones y las **métricas** del resultado (puntos, posición, vueltas, tiempos, etc.).

```text
                 dim_drivers
                      |
dim_constructors -- fact_results -- dim_status
                      |
                  dim_races
           (circuito + temporada embebidos)
```

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("f1db-star-fact-results")
    .master("local[2]")
    .config("spark.driver.memory", "3g")
    .config("spark.driver.cores", "2")
    .config("spark.jars", "/opt/spark/jars/mysql-connector-j-8.4.0.jar")
    .getOrCreate()
)

spark

In [2]:
JDBC_URL = (
    "jdbc:mysql://mysql:3306/f1db"
    "?useSSL=false"
    "&allowPublicKeyRetrieval=true"
    "&serverTimezone=UTC"
)

JDBC_OPTS = {
    "url": JDBC_URL,
    "user": "f1user",
    "password": "f1password",
    "driver": "com.mysql.cj.jdbc.Driver",
}


def read_table(table_name: str):
    return (
        spark.read.format("jdbc")
        .options(**JDBC_OPTS)
        .option("dbtable", table_name)
        .load()
    )


def spark_sql(query: str, mapping: dict, view_name: str):
    """Registra DataFrames como vistas temporales y ejecuta SQL."""
    for alias, frame in mapping.items():
        frame.createOrReplaceTempView(alias)
    result = spark.sql(query)
    result.createOrReplaceTempView(view_name)
    return result


results = read_table("results")
drivers = read_table("drivers")
constructors = read_table("constructors")
status = read_table("status")
races = read_table("races")
circuits = read_table("circuits")
seasons = read_table("seasons")

print("Filas origen")
for name, df in [
    ("results", results),
    ("drivers", drivers),
    ("constructors", constructors),
    ("status", status),
    ("races", races),
    ("circuits", circuits),
    ("seasons", seasons),
]:
    print(f"  {name:14} {df.count():>8}")

Filas origen
  results           26759
  drivers             861
  constructors        212
  status              139
  races              1125
  circuits             77
  seasons              75


## `dim_drivers`

Atributos del piloto. `driverId` es la clave de negocio que usa el hecho.

In [3]:
sql_dim_drivers = """
with dim_drivers as (
    select
        driverId,
        driverRef,
        number as permanentNumber,
        code,
        forename,
        surname,
        concat(forename, ' ', surname) as fullName,
        dob,
        nationality,
        url as wikipediaUrl
    from drivers
)
select * from dim_drivers
"""

dim_drivers = spark_sql(
    sql_dim_drivers,
    mapping={"drivers": drivers},
    view_name="dim_drivers",
)

dim_drivers.show(10, truncate=False)
print(f"dim_drivers: {dim_drivers.count()} filas")

+--------+----------+---------------+----+---------+----------+------------------+----------+-----------+-----------------------------------------------------+
|driverId|driverRef |permanentNumber|code|forename |surname   |fullName          |dob       |nationality|wikipediaUrl                                         |
+--------+----------+---------------+----+---------+----------+------------------+----------+-----------+-----------------------------------------------------+
|1       |hamilton  |44             |HAM |Lewis    |Hamilton  |Lewis Hamilton    |1985-01-07|British    |http://en.wikipedia.org/wiki/Lewis_Hamilton          |
|2       |heidfeld  |null           |HEI |Nick     |Heidfeld  |Nick Heidfeld     |1977-05-10|German     |http://en.wikipedia.org/wiki/Nick_Heidfeld           |
|3       |rosberg   |6              |ROS |Nico     |Rosberg   |Nico Rosberg      |1985-06-27|German     |http://en.wikipedia.org/wiki/Nico_Rosberg            |
|4       |alonso    |14             |ALO

## `dim_constructors`

Escudería / constructor del auto en ese resultado.

In [4]:
sql_dim_constructors = """
with dim_constructors (
    select
        constructorId,
        constructorRef,
        name as constructorName,
        nationality,
        url as wikipediaUrl
    from constructors
)
select * from dim_constructors
"""

dim_constructors = spark_sql(
    sql_dim_constructors,
    mapping={"constructors": constructors},
    view_name="dim_constructors",
)

dim_constructors.show(10, truncate=False)
print(f"dim_constructors: {dim_constructors.count()} filas")

+-------------+--------------+---------------+-----------+------------------------------------------------------------+
|constructorId|constructorRef|constructorName|nationality|wikipediaUrl                                                |
+-------------+--------------+---------------+-----------+------------------------------------------------------------+
|1            |mclaren       |McLaren        |British    |http://en.wikipedia.org/wiki/McLaren                        |
|2            |bmw_sauber    |BMW Sauber     |German     |http://en.wikipedia.org/wiki/BMW_Sauber                     |
|3            |williams      |Williams       |British    |http://en.wikipedia.org/wiki/Williams_Grand_Prix_Engineering|
|4            |renault       |Renault        |French     |http://en.wikipedia.org/wiki/Renault_in_Formula_One         |
|5            |toro_rosso    |Toro Rosso     |Italian    |http://en.wikipedia.org/wiki/Scuderia_Toro_Rosso            |
|6            |ferrari       |Ferrari   

## `dim_status`

Por qué el resultado quedó así: terminó, se retiró, descalificado, etc.
`Finished` es el caso normal; el resto explica abandonos y excepciones.

In [5]:
sql_dim_status = """
with dim_status as (
    select
        statusId,
        status
    from status
)
select * from dim_status
"""

dim_status = spark_sql(
    sql_dim_status,
    mapping={"status": status},
    view_name="dim_status",
)

dim_status.show(20, truncate=False)
print(f"dim_status: {dim_status.count()} filas")

+--------+------------+
|statusId|status      |
+--------+------------+
|1       |Finished    |
|2       |Disqualified|
|3       |Accident    |
|4       |Collision   |
|5       |Engine      |
|6       |Gearbox     |
|7       |Transmission|
|8       |Clutch      |
|9       |Hydraulics  |
|10      |Electrical  |
|11      |+1 Lap      |
|12      |+2 Laps     |
|13      |+3 Laps     |
|14      |+4 Laps     |
|15      |+5 Laps     |
|16      |+6 Laps     |
|17      |+7 Laps     |
|18      |+8 Laps     |
|19      |+9 Laps     |
|20      |Spun off    |
+--------+------------+
only showing top 20 rows

dim_status: 139 filas


## `dim_races`

Una fila por Gran Premio. El circuito y la temporada se pegan acá para no snowflakear.

- `races.year` ya es la temporada; `seasons` solo aporta la URL de Wikipedia.
- `circuits` aporta nombre, país, coordenadas y altitud.

In [6]:
sql_dim_races = """
with dim_races as (
    select
        races.raceId,
        races.year,
        races.round,
        races.name as raceName,
        races.date as raceDate,
        races.time as raceStartTime,
        races.url as raceWikipediaUrl,
        seasons.url as seasonWikipediaUrl,
        circuits.circuitId,
        circuits.circuitRef,
        circuits.name as circuitName,
        circuits.location as circuitLocation,
        circuits.country as circuitCountry,
        circuits.lat as circuitLat,
        circuits.lng as circuitLng,
        circuits.alt as circuitAltitudeM
    from races
    left join circuits using (circuitId)
    left join seasons using (year)
)
select * from dim_races
"""

dim_races = spark_sql(
    sql_dim_races,
    mapping={
        "races": races,
        "circuits": circuits,
        "seasons": seasons,
    },
    view_name="dim_races",
)

dim_races.show(10, truncate=False)
print(f"dim_races: {dim_races.count()} filas")

+------+----+-----+---------------------+----------+-------------------+-------------------------------------------------------+----------------------------------------------------+---------+-----------+------------------------------+---------------+--------------+----------+----------+----------------+
|raceId|year|round|raceName             |raceDate  |raceStartTime      |raceWikipediaUrl                                       |seasonWikipediaUrl                                  |circuitId|circuitRef |circuitName                   |circuitLocation|circuitCountry|circuitLat|circuitLng|circuitAltitudeM|
+------+----+-----+---------------------+----------+-------------------+-------------------------------------------------------+----------------------------------------------------+---------+-----------+------------------------------+---------------+--------------+----------+----------+----------------+
|1     |2009|1    |Australian Grand Prix|2009-03-29|1970-01-01 06:00:00|http://en.wik

## `fact_results`

Grano: **un resultado de un piloto en una carrera** (`resultId`).

Claves foráneas: `raceId`, `driverId`, `constructorId`, `statusId`.

Métricas:

- `grid`, `position`, `positionOrder`, `points`, `laps`
- `milliseconds` (tiempo de carrera)
- `fastestLap`, `rank`, `fastestLapTime`, `fastestLapSpeed`

Atributos degenerados (viven en el hecho porque no merecen dimensión propia):

- `number` (número de auto en esa carrera; no es el permanente del piloto)
- `positionText` (`1`, `R`, `D`, etc.)
- `time` (texto del gap o tiempo de llegada)

In [7]:
sql_fact_results = """
with fact_results as (
    select
        resultId,
        raceId,
        driverId,
        constructorId,
        statusId,
        number as raceNumber,
        grid,
        position,
        positionText,
        positionOrder,
        points,
        laps,
        time as finishTimeText,
        milliseconds as finishTimeMs,
        fastestLap,
        rank as fastestLapRank,
        fastestLapTime,
        fastestLapSpeed
    from results
)
select * from fact_results
"""

fact_results = spark_sql(
    sql_fact_results,
    mapping={"results": results},
    view_name="fact_results",
)

fact_results.show(10, truncate=False)
print(f"fact_results: {fact_results.count()} filas")

+--------+------+--------+-------------+--------+----------+----+--------+------------+-------------+------+----+--------------+------------+----------+--------------+--------------+---------------+
|resultId|raceId|driverId|constructorId|statusId|raceNumber|grid|position|positionText|positionOrder|points|laps|finishTimeText|finishTimeMs|fastestLap|fastestLapRank|fastestLapTime|fastestLapSpeed|
+--------+------+--------+-------------+--------+----------+----+--------+------------+-------------+------+----+--------------+------------+----------+--------------+--------------+---------------+
|1       |18    |1       |1            |1       |22        |1   |1       |1           |1            |10.0  |58  |1:34:50.616   |5690616     |39        |2             |1:27.452      |218.300        |
|2       |18    |2       |2            |1       |3         |5   |2       |2           |2            |8.0   |58  |+5.478        |5696094     |41        |3             |1:27.739      |217.586        |
|3   

## Prueba del modelo

Consulta típica de estrella: el hecho se une a las cuatro dimensiones por clave, sin pasar por tablas puente.

In [8]:
star_check = spark.sql("""
select
    r.year,
    r.round,
    r.raceName,
    r.circuitName,
    r.circuitCountry,
    d.fullName as driver,
    c.constructorName,
    f.grid,
    f.positionOrder,
    f.points,
    s.status
from fact_results f
inner join dim_races r using (raceId)
inner join dim_drivers d using (driverId)
inner join dim_constructors c using (constructorId)
inner join dim_status s using (statusId)
where r.year = 2024
  and r.round = 1
order by f.positionOrder
""")

star_check.show(30, truncate=False)

+----+-----+------------------+-----------------------------+--------------+----------------+---------------+----+-------------+------+--------+
|year|round|raceName          |circuitName                  |circuitCountry|driver          |constructorName|grid|positionOrder|points|status  |
+----+-----+------------------+-----------------------------+--------------+----------------+---------------+----+-------------+------+--------+
|2024|1    |Bahrain Grand Prix|Bahrain International Circuit|Bahrain       |Max Verstappen  |Red Bull       |1   |1            |26.0  |Finished|
|2024|1    |Bahrain Grand Prix|Bahrain International Circuit|Bahrain       |Sergio Pérez    |Red Bull       |5   |2            |18.0  |Finished|
|2024|1    |Bahrain Grand Prix|Bahrain International Circuit|Bahrain       |Carlos Sainz    |Ferrari        |4   |3            |15.0  |Finished|
|2024|1    |Bahrain Grand Prix|Bahrain International Circuit|Bahrain       |Charles Leclerc |Ferrari        |2   |4            |12